<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/document-intelligence/lessons/P02-L11-capstone-conformity-pack/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/document-intelligence/lessons/P02-L11-capstone-conformity-pack/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/document-intelligence/lessons/P02-L11-capstone-conformity-pack/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/document-intelligence/lessons/P02-L11-capstone-conformity-pack/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P02-L11 · The capstone: the conformity pack

**You will build:** the validator that decides whether an evidence pack is *documented* —
a function that regenerates every number in the pack from the artefacts behind it, and
refuses the pack the moment one figure does not come back the same way twice.

**Time:** ~100 minutes · **Runs on:** a laptop CPU, no download, no network ·
**Prerequisites:** the earlier modules of this programme. The extraction harness
(module 1), the review policy (module 6), slice analysis (module 7), the drift monitor
(module 8) and the cost model (module 9) are all carried forward below, in the minimal
faithful form this notebook needs — this lesson opens on its own and may not import
another lesson's files.

The pipeline itself is **synthetic and generated in this notebook**, from a fixed seed.
No real vendor, no real invoice, no real reviewer.

By the end you will be able to:

1. Implement a generic reader that finds every figure a document prints, and tells a
   figure apart from a date, a heading number or an identifier.
2. Implement a generic collector that finds every computed result a pipeline produced,
   however deeply it is nested, and ignores what only looks like a result.
3. Implement a reproduction gate that fails a document the moment one printed figure has
   no computed result behind it, at the precision the document itself claimed.
4. Implement three independent checks that catch a documented pack whose numbers are
   individually correct but whose *procedure* was not: a metric measured on the wrong
   split, a cost figure with nothing measuring it, and a threshold tuned with hindsight.
5. Assemble all four into one validator, and explain why "the pack is green" and "the pack
   is documented" are different claims — and why only one of them survives a repair that
   merely makes the checker quieter.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import random
import re
import sys
import traceback
from typing import Any, Callable, Mapping, NamedTuple, Sequence

import numpy as np

SEED = 20260923           # this pipeline's corpus, bootstrap and drift replay
N_DOCS = 240
FIELD_TYPES = ("money", "date", "id", "counterparty")
AS_OF = "2026-09-23"       # this pack's stated date. Every age or window is measured against
                           # it, never against the clock — GATE 13 forbids reading today's date.

_LESSON_T0 = 0.0
print("python", sys.version.split()[0], "· numpy", np.__version__)

DATA_NOTE = (
    "SYNTHETIC DATA. Every record below was generated inside this notebook by "
    f"random.Random({SEED}). No real vendor, invoice or reviewer is represented."
)
print(DATA_NOTE)

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the function each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell waiting on
# an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("extract_figures",),
    "exercise 2": ("computed_values",),
    "exercise 3": ("trace_gate",),
    "exercise 4": ("check_residual_provenance",),
    "exercise 5": ("check_cost_provenance",),
    "exercise 6": ("check_threshold_timing",),
    "exercise 7": ("validate_pack",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (trace_gate)"; several -> "exercises 3, 4 and 7"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The pipeline this pack documents, carried forward from module 1

A synthetic extraction pipeline reads four field types off short remittance-advice-style
documents and returns a predicted value for each. `normalise_value`, `match_value`,
`score_field` and `macro_f1` are module 1's exact-match harness, carried forward
unchanged: a wrong value counts as both a false positive and a false negative, and macro
F1 is the unweighted mean of the per-field F1 scores. Nothing in this section is an
exercise — you built this once already.

`build_corpus` generates the documents. A third of them are held out as the **evaluation
split** and never seen by anything that tunes the pipeline; the rest are the **training
split**. This generator makes the evaluation split harder on purpose (`EVAL_ERROR_MULT`),
standing in for a pipeline that does better on the data it was tuned on than on data it
has never seen — so a figure measured on the wrong split flatters it, and section 3 is
built around exactly that.

Two things outside this notebook motivate this lesson's shape. A survey of research
fields that adopted machine learning found data-leakage errors in 17 of them, and sorted
leakage into a taxonomy running from textbook errors to open research problems (Kapoor
and Narayanan, arXiv 2207.07048, 2022). And the case for a released model being
accompanied by documentation of its measured performance, rather than an assertion of
it, is the original case for a model card (Mitchell et al., 2019). Both are sourced in
`claims.yaml`. This lesson's second planted defect — a metric taken from the split the
pipeline was tuned on — is a textbook instance of the first, planted on purpose, and the
pack itself is this course's small instance of the second.

In [ ]:
class FieldScore(NamedTuple):
    field_type: str
    tp: int
    fp: int
    fn: int
    precision: float
    recall: float
    f1: float


def normalise_value(value: str, field_type: str) -> str:
    """Module 1's per-field-type normaliser, carried forward unchanged."""
    v = value.strip()
    if field_type == "money":
        m = re.match(r"^\$?([\d,]+)\.(\d{2})$", v.replace(" ", ""))
        if not m:
            return v.lower()
        return f"{int(m.group(1).replace(',', '')):d}.{m.group(2)}"
    if field_type == "date":
        return v
    if field_type == "id":
        return re.sub(r"[^A-Za-z0-9]", "", v).upper()
    return v.strip().lower()


def match_value(pred: str, gold: str, field_type: str) -> bool:
    """Module 1's matcher: exact match after normalisation."""
    return normalise_value(pred, field_type) == normalise_value(gold, field_type)


def score_field(records: Sequence[Mapping], field_type: str) -> FieldScore:
    """Module 1's per-field scorer: a wrong value is both a false positive and a false
    negative."""
    rows = [r for r in records if r["field_type"] == field_type]
    tp = sum(1 for r in rows if match_value(r["pred"], r["gold"], field_type))
    wrong = len(rows) - tp
    fp = fn = wrong
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return FieldScore(field_type, tp, fp, fn, precision, recall, f1)


def macro_f1(records: Sequence[Mapping]) -> float:
    """Module 1's headline: the unweighted mean of the per-field-type F1 scores."""
    types = sorted({r["field_type"] for r in records})
    return sum(score_field(records, ft).f1 for ft in types) / len(types)


_NAMES = ("Northwind Freight", "Blue Harbor Logistics", "Cascade Mills", "Iron Gate Supply",
          "Union Pacific Trading", "Silverline Components")
BASE_ERROR_RATE = {"money": 0.05, "date": 0.04, "id": 0.07, "counterparty": 0.09}
EVAL_ERROR_MULT = 1.8      # the evaluation split is genuinely harder than the training split


def _surface_gold(field_type: str, rng: random.Random) -> str:
    if field_type == "money":
        return f"{rng.randint(10, 9999)}.{rng.randint(0, 99):02d}"
    if field_type == "date":
        return f"{rng.choice((2025, 2026))}-{rng.randint(1, 12):02d}-{rng.randint(1, 28):02d}"
    if field_type == "id":
        return f"INV-{rng.randint(1000, 9999)}"
    return rng.choice(_NAMES)


def _corrupt(gold: str, field_type: str, rng: random.Random) -> str:
    if field_type == "money":
        whole, cents = gold.split(".")
        bump = rng.choice((-1, 1)) * rng.randint(1, 9)
        return f"{max(0, int(whole) + bump)}.{cents}"
    if field_type == "date":
        y, m, d = gold.split("-")
        return f"{y}-{m}-{max(1, min(28, int(d) + rng.choice((-1, 1)))):02d}"
    if field_type == "id":
        digits = gold[4:]
        pos = rng.randrange(len(digits))
        bumped = str((int(digits[pos]) + rng.randint(1, 9)) % 10)
        return "INV-" + digits[:pos] + bumped + digits[pos + 1:]
    return rng.choice([n for n in _NAMES if n != gold])


def build_corpus(n_docs: int = N_DOCS, seed: int = SEED) -> list[dict]:
    """Module 1's corpus generator, carried forward and extended with `split`. SYNTHETIC."""
    rng = random.Random(seed)
    rows = []
    for i in range(n_docs):
        doc_id = f"D{i:04d}"
        split = "eval" if i % 3 == 0 else "train"
        for field_type in FIELD_TYPES:
            gold = _surface_gold(field_type, rng)
            rate = BASE_ERROR_RATE[field_type] * (EVAL_ERROR_MULT if split == "eval" else 1.0)
            pred = _corrupt(gold, field_type, rng) if rng.random() < rate else gold
            rows.append({"doc_id": doc_id, "field_type": field_type, "gold": gold,
                         "pred": pred, "split": split})
    return rows


RECORDS = build_corpus()
EVAL_RECORDS = [r for r in RECORDS if r["split"] == "eval"]
TRAIN_RECORDS = [r for r in RECORDS if r["split"] == "train"]
N_EVAL_DOCS = len({r["doc_id"] for r in EVAL_RECORDS})
N_TRAIN_DOCS = len({r["doc_id"] for r in TRAIN_RECORDS})
EVAL_MACRO_F1 = macro_f1(EVAL_RECORDS)
TRAIN_MACRO_F1 = macro_f1(TRAIN_RECORDS)
print(f"{N_DOCS} documents · {N_EVAL_DOCS} eval / {N_TRAIN_DOCS} train")
print(f"macro F1 — eval: {EVAL_MACRO_F1:.4f} · train: {TRAIN_MACRO_F1:.4f}")

## 2. Three more measurements, carried forward

**Residual error by class (modules 1 and 7).** `residual_by_class` combines module 1's
scorer with module 7's paired bootstrap: the per-field-type error rate on one named
split, with a percentile interval built by resampling *documents* with replacement — not
rows, because a document's four fields are not independent draws. The point estimate is
computed once, directly, on the named split; only the interval is resampled. Given, not
graded — you built the bootstrap once already, in module 7.

In [ ]:
class Interval(NamedTuple):
    point: float
    lo: float
    hi: float
    se: float


N_BOOT = 300
CI_ALPHA = 0.05


def residual_by_class(records: Sequence[Mapping], split: str, n_boot: int = N_BOOT,
                      alpha: float = CI_ALPHA, seed: int = SEED) -> dict[str, Interval]:
    """Modules 1 and 7, assembled: per-field-type residual error on one split, with a
    document-level percentile bootstrap interval. GIVEN, not graded."""
    rows = [r for r in records if r["split"] == split]
    by_doc: dict[str, list[dict]] = {}
    for r in rows:
        by_doc.setdefault(r["doc_id"], []).append(r)
    doc_ids = sorted(by_doc)
    rng = random.Random(seed)
    out: dict[str, Interval] = {}
    for field_type in FIELD_TYPES:
        type_rows = [r for r in rows if r["field_type"] == field_type]
        total = len(type_rows)
        wrong = sum(1 for r in type_rows if not match_value(r["pred"], r["gold"], field_type))
        point = wrong / total if total else 0.0
        reps = []
        for _ in range(n_boot):
            sample = [rng.choice(doc_ids) for _ in doc_ids]
            sample_rows = [r for d in sample for r in by_doc[d] if r["field_type"] == field_type]
            w = sum(1 for r in sample_rows if not match_value(r["pred"], r["gold"], field_type))
            reps.append(w / len(sample_rows) if sample_rows else 0.0)
        reps.sort()
        lo = reps[int((alpha / 2) * n_boot)]
        hi = reps[min(n_boot - 1, int((1 - alpha / 2) * n_boot))]
        mean = sum(reps) / len(reps)
        var = sum((x - mean) ** 2 for x in reps) / (len(reps) - 1) if len(reps) > 1 else 0.0
        out[field_type] = Interval(point, lo, hi, math.sqrt(var))
    return out


RESIDUAL_EVAL = residual_by_class(RECORDS, "eval")
RESIDUAL_TRAIN = residual_by_class(RECORDS, "train")
for _ft in FIELD_TYPES:
    print(f"{_ft:<13} eval {RESIDUAL_EVAL[_ft].point:.4f} "
          f"({RESIDUAL_EVAL[_ft].lo:.4f}-{RESIDUAL_EVAL[_ft].hi:.4f})  ·  "
          f"train {RESIDUAL_TRAIN[_ft].point:.4f} "
          f"({RESIDUAL_TRAIN[_ft].lo:.4f}-{RESIDUAL_TRAIN[_ft].hi:.4f})")

**The cost model (module 9).** Three tiers, each with a cost and an accuracy: a cheap
extractor, an expensive extractor, and a human reviewer. `MEASUREMENTS` is the
registry every *measured* cost figure in the pack must cite — a named source, a date and
a unit, synthetic like everything else here — and it is what section 8's check reads
from, not the pack's own say-so. Given, not graded.

In [ ]:
class Tier(NamedTuple):
    name: str
    cost_per_doc: float
    accuracy: float


TIERS = (Tier("cheap_extractor", 0.02, 0.78), Tier("expensive_extractor", 0.11, 0.91),
         Tier("human_review", 1.40, 0.99))
ROUTING_SHARE = {"cheap_extractor": 0.70, "expensive_extractor": 0.24, "human_review": 0.06}
MONTHLY_VOLUME = 50_000.0


class Measurement(NamedTuple):
    measurement_id: str
    description: str
    value: float
    unit: str


MEASUREMENTS: dict[str, Measurement] = {
    "cost.cheap.rate": Measurement("cost.cheap.rate",
                                   "cheap extractor, vendor invoice, September 2026",
                                   0.02, "$/doc"),
    "cost.expensive.rate": Measurement("cost.expensive.rate",
                                       "expensive extractor, vendor invoice, September 2026",
                                       0.11, "$/doc"),
    "cost.human.rate": Measurement("cost.human.rate",
                                   "reviewer cost per document, Q3 2026 timesheets",
                                   1.40, "$/doc"),
    "cost.volume.monthly": Measurement("cost.volume.monthly",
                                       "documents processed, trailing 30 days, pipeline logs",
                                       50_000.0, "docs/month"),
}


class CostFigure(NamedTuple):
    label: str
    value: float
    kind: str          # "measured" (cites MEASUREMENTS) or "derived" (computed from tiers)
    source_id: str      # a key into MEASUREMENTS; "" only ever allowed for "derived"


def blended_cost_per_doc(tiers: Sequence[Tier], shares: Mapping[str, float]) -> float:
    """Module 9's routing-mix cost: a share-weighted sum. GIVEN, not graded."""
    return sum(shares.get(t.name, 0.0) * t.cost_per_doc for t in tiers)


BLENDED_COST = blended_cost_per_doc(TIERS, ROUTING_SHARE)
print(f"blended cost per document: {BLENDED_COST:.4f}")

**The drift monitor (module 8).** A one-sided CUSUM over a daily quality-proxy series,
with one genuine regression injected partway through the replay. `Threshold` records not
just `k` and `h` but the last day of data the threshold was *calibrated* against —
because a threshold is only evidence of anything if it was fixed before the day it is
used to explain. Given, not graded; section 9 is the exercise that reads
`calibrated_through_day` for the first time.

In [ ]:
class CusumResult(NamedTuple):
    alarm_day: int | None
    peak: float


def cusum(z: Sequence[float], k: float, h: float) -> CusumResult:
    """Module 8's one-sided CUSUM, carried forward unchanged."""
    pos = peak = 0.0
    alarm = None
    for day, value in enumerate(z, start=1):
        pos = max(0.0, pos + value - k)
        peak = max(peak, pos)
        if alarm is None and pos > h:
            alarm = day
    return CusumResult(alarm, peak)


class Threshold(NamedTuple):
    k: float
    h: float
    calibrated_through_day: int


N_DAYS = 300
REGRESSION_DAY = 250       # the one genuine regression in this replay
K, H = 0.5, 6.0


def simulate_quality_series(seed: int = SEED, n_days: int = N_DAYS) -> tuple:
    """A daily quality-proxy z-score series: noise, plus one genuine regression starting at
    REGRESSION_DAY. GIVEN — the synthetic replay this pack's drift section monitors."""
    rng = random.Random(seed + 8)
    return tuple(rng.gauss(0.0, 1.0) + (1.4 if day >= REGRESSION_DAY else 0.0)
                for day in range(1, n_days + 1))


Z_SERIES = simulate_quality_series()
ALARM = cusum(Z_SERIES, K, H)
assert ALARM.alarm_day is not None and ALARM.alarm_day >= REGRESSION_DAY, (
    "the replay must alarm, and only after the injected regression — tune K, H or the "
    "injected shift if this ever fires")
print(f"CUSUM alarm on day {ALARM.alarm_day} (regression injected at day {REGRESSION_DAY}), "
      f"peak {ALARM.peak:.4f}")

**Oversight (module 6).** Under the escalation policy, any document with at least one
field the extractor got wrong is routed to a human reviewer — the simplest possible
policy, and the one this pack reports. Given, not graded.

In [ ]:
def escalation_rate(records: Sequence[Mapping]) -> float:
    """Module 6: the share of documents with at least one wrong field. GIVEN, not graded."""
    by_doc: dict[str, list[dict]] = {}
    for r in records:
        by_doc.setdefault(r["doc_id"], []).append(r)
    escalated = sum(1 for rows in by_doc.values()
                    if any(not match_value(r["pred"], r["gold"], r["field_type"]) for r in rows))
    return escalated / len(by_doc)


ESCALATION_RATE = escalation_rate(EVAL_RECORDS)
print(f"escalation rate on the eval split: {ESCALATION_RATE:.4f}")

## 3. The pack you are handed

`_pack_body` formats a conformity pack from whatever is handed to it. It checks nothing —
it is exactly as trustworthy as whoever called it. `draft_pack_with_defects` calls it with
four things wrong, alongside a `claimed_evidence` bundle that is a completely honest
record of what *was* computed — every number below traces to something in it. That is
the trap: three of the four defects are not typos, they are correct numbers computed the
wrong way, and a gate that only checks "does this figure appear somewhere" will wave
every one of them through.

In [ ]:
def _residual_lines(residual: Mapping[str, Interval], split_label: str) -> list[str]:
    lines = [f"Residual error by field type, on the {split_label} split, with a bootstrap "
             "interval over documents:", "",
             "| field type | residual error | interval |", "| --- | --- | --- |"]
    for ft in FIELD_TYPES:
        iv = residual[ft]
        lines.append(f"| {ft} | {iv.point:.4f} | {iv.lo:.4f}-{iv.hi:.4f} |")
    return lines


def _cost_lines(cost_figures: Sequence[CostFigure]) -> list[str]:
    lines = ["| figure | value | kind | source |", "| --- | --- | --- | --- |"]
    for fig in cost_figures:
        lines.append(f"| {fig.label} | {fig.value:.4f} | {fig.kind} | "
                     f"{fig.source_id or '(none)'} |")
    return lines


def _pack_body(residual: Mapping[str, Interval], split_label: str,
              cost_figures: Sequence[CostFigure], threshold: Threshold, alarm_day: int,
              commentary: str) -> str:
    """GIVEN: the pack's text, exactly as its preparer assembled it. Formats every figure
    from its arguments and checks nothing. Shared by the honest artefact function
    (`render_conformity_pack`, which gates it) and the demonstration draft below (which
    does not)."""
    lines = [
        "# Conformity pack -- synthetic remittance-advice extraction pipeline", "",
        f"As of {AS_OF}. Prepared as evidence for release review.", "",
        "## 1. What it does", "",
        "A synthetic extraction pipeline reads four field types off short "
        "remittance-advice-style documents and returns a predicted value for each, scored "
        "against the gold value with an exact-match rule after normalisation.", "",
        "## 2. How it was measured, on what data", "",
        f"{N_DOCS} synthetic documents were generated from a fixed seed; {N_EVAL_DOCS} of "
        "them were held out as the evaluation split and never used to tune anything. The "
        f"headline macro F1 on that split is {EVAL_MACRO_F1:.4f}.", "",
        "## 3. Residual error by class", "",
        *_residual_lines(residual, split_label), "",
        "## 4. Drift and oversight", "",
        f"A daily quality-proxy CUSUM (k={threshold.k:.2f}, h={threshold.h:.2f}) monitors "
        f"the live pipeline. It alarmed on day {alarm_day}, peaking at {ALARM.peak:.4f}. "
        "The alarm threshold was calibrated using data through day "
        f"{threshold.calibrated_through_day}. Under the escalation policy, "
        f"{ESCALATION_RATE:.4f} of evaluation documents were routed to a human reviewer.", "",
        "## 5. Cost", "",
        *_cost_lines(cost_figures),
        "", f"Monthly volume: {MONTHLY_VOLUME:,.0f} documents.", "",
        "## 6. Commentary", "", commentary or "None.", "",
    ]
    return "\n".join(lines)


def _build_claimed_evidence(residual: Mapping[str, Interval], cost_figures: Sequence[CostFigure],
                            threshold: Threshold, alarm_day: int) -> dict:
    """An honest record of what was actually computed for THIS pack — self-consistent with
    whatever `_pack_body` was handed, whether or not the procedure behind it was sound.
    GIVEN. This is deliberately not the same thing as "correct": section 7 recomputes the
    residual table independently rather than trusting this bundle, and sections 8 and 9
    check things this bundle cannot speak to at all."""
    return {
        "n_docs": N_DOCS, "n_eval_docs": N_EVAL_DOCS, "macro_f1": EVAL_MACRO_F1,
        "residual": dict(residual), "cost_figures": tuple(cost_figures),
        "monthly_volume": MONTHLY_VOLUME, "threshold": threshold, "alarm_day": alarm_day,
        "peak": ALARM.peak, "escalation_rate": ESCALATION_RATE,
    }


CLEAN_COST_FIGURES = (
    CostFigure("cheap extractor rate", TIERS[0].cost_per_doc, "measured", "cost.cheap.rate"),
    CostFigure("expensive extractor rate", TIERS[1].cost_per_doc, "measured",
              "cost.expensive.rate"),
    CostFigure("human review rate", TIERS[2].cost_per_doc, "measured", "cost.human.rate"),
    CostFigure("blended cost per document", BLENDED_COST, "derived", ""),
)
CLEAN_THRESHOLD = Threshold(K, H, REGRESSION_DAY - 20)
CLEAN_RESIDUAL, CLEAN_SPLIT_LABEL = RESIDUAL_EVAL, "eval"
CLEAN_COMMENTARY = "No open items."

DEFECT_COST_FIGURES = (
    CostFigure("cheap extractor rate", TIERS[0].cost_per_doc, "measured", "cost.cheap.rate"),
    CostFigure("expensive extractor rate", TIERS[1].cost_per_doc, "measured",
              "cost.expensive.rate"),
    CostFigure("human review rate", TIERS[2].cost_per_doc, "measured", "cost.human.rate"),
    CostFigure("blended cost per document", BLENDED_COST, "measured", ""),   # defect 3
)
DEFECT_THRESHOLD = Threshold(K, H, ALARM.alarm_day + 5)                       # defect 4
DEFECT_RESIDUAL, DEFECT_SPLIT_LABEL = RESIDUAL_TRAIN, "eval"                  # defect 2 (a lie)
DEFECT_COMMENTARY = "Coverage on this run was 96.7%, comfortably above policy."  # defect 1


def draft_pack_with_defects() -> str:
    """GIVEN: the pack as its preparer assembled it, with four things wrong. Ungated — this
    is what lands in your inbox, not what this lesson's own artefact function would ever
    produce."""
    return _pack_body(DEFECT_RESIDUAL, DEFECT_SPLIT_LABEL, DEFECT_COST_FIGURES,
                      DEFECT_THRESHOLD, ALARM.alarm_day, DEFECT_COMMENTARY)


def draft_claimed_evidence_with_defects() -> dict:
    """GIVEN: exactly what the preparer of `draft_pack_with_defects` says they computed."""
    return _build_claimed_evidence(DEFECT_RESIDUAL, DEFECT_COST_FIGURES, DEFECT_THRESHOLD,
                                   ALARM.alarm_day)


DRAFT = draft_pack_with_defects()
print(DRAFT)

Four things are wrong with it. A careful reader might spot one or two; nothing in the
text says the residual table came from the training split, and no amount of rereading
will find that. The rest of this lesson builds four checks that do not depend on a
careful reader.

## 4. Exercise 1 — `extract_figures()`

The first thing a validator needs is a precise idea of what counts as a figure, because
"every number must trace" is not a rule until you can say which strings are numbers. A
date is not a result. Neither is the section number on a heading, or the `3` in
`INV-1003`. Get the definition too narrow and a typed number walks straight through.

The definition this lesson uses, which your function implements exactly:

- **Not figures:** a date written `YYYY-MM-DD`; the number straight after the `#`s that
  open a markdown heading line; any digit that sits inside a *word* — a maximal run of
  `[A-Za-z0-9_.+-]` — that also contains a letter (`D0007`, `INV-1234`, `cost.cheap.rate`
  has none, `F-003` would be masked if it appeared).
- **How a figure is read:** `1,234` is one figure, its comma grouped in exactly threes;
  a leading `-` or `+` is a sign only when the character immediately before it is not
  itself a digit — so in `0.0041-0.0512` the dash is a range and both figures are
  positive, but in `= -0.0123` the minus is a sign. A trailing `%` divides the value by
  100 and travels with the figure.
- **How precise it claims to be:** a figure printed to *d* decimal places claims its
  value to within half a unit in that last place — `half_unit`. That is the only tolerance
  a document itself ever asserts, so it is the only one a gate may use.

<details><summary>💡 Hint 1 — what to think about</summary>

Work each line in two passes. First mark the spans that are names, not figures: every
date, the heading's leading token, and every alphanumeric word that mixes a letter with a
digit. Second, scan for numbers, and drop any match that starts inside a masked span. The
sign and the percent sign are decisions about the single character on either side of the
digits, not about the whole line.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Number the lines from 1. Build the masked spans with three regular expressions: dates,
the first token of a heading line, and alphanumeric words containing at least one letter
and one digit. Scan for numbers with a pattern that requires the character immediately
before the match not to be a digit (so a range's dash is never captured as a sign),
allows digits with comma groups of exactly three OR a plain run of digits, an optional
`.` fraction and an optional trailing `%`. Skip any match whose start falls inside a
masked span. Strip commas and the `%` to get the numeric text; the sign, if present,
negates it; the `%` divides it by 100. `half_unit` is half a unit in the last printed
decimal place, also divided by 100 for a percent figure.

</details>

In [ ]:
class Figure(NamedTuple):
    text: str          # exactly as printed: "1,234", "22.6%", "-0.0123"
    value: float        # what it means: 1234.0, 0.226, -0.0123
    half_unit: float    # half a unit in its last printed place, on the same scale as value
    line: int            # 1-based line number in the document


_DATE_RE = re.compile(r"\b\d{4}-\d{2}-\d{2}\b")
_HEADING_RE = re.compile(r"^#+\s+(\S+)")
_WORD_RE = re.compile(r"[A-Za-z0-9_.+-]+")
_NUMBER_RE = re.compile(r"(?<![\d.])[+-]?\d+(?:,\d{3})*(?:\.\d+)?%?")


def extract_figures(text: str) -> tuple:
    """Every figure in `text`, under this course's definition, in document order.

    Requirements, each graded:
      * NOT figures: a `YYYY-MM-DD` date; the token straight after the `#`s of a markdown
        heading line ("## 3." -> "3." is masked); any digit inside a word — a maximal run
        of `[A-Za-z0-9_.+-]` — that also contains a letter.
      * a figure is: an optional sign, then digits (with `,` only between groups of
        exactly three), an optional `.` fraction, and an optional trailing `%`.
      * the sign counts only when the character immediately before it is not a digit: in
        "0.00-0.10" both figures are positive; in "= -0.0123" the figure is negative.
      * `value`: commas removed, sign applied, divided by 100 for a `%`.
      * `half_unit`: 0.5 * 10 ** -decimals, where decimals counts the digits after the
        point (0 for an integer); divided by 100 for a `%`. So "0.0727" -> 5e-05,
        "6,000" -> 0.5, "22.6%" -> 0.0005.
      * `line` is 1-based; figures come back in document order as a tuple of Figure.

    Example:
        >>> [(f.text, f.value) for f in extract_figures("D0007 on 2026-09-23: 1,234, 12.5%")]
        [('1,234', 1234.0), ('12.5%', 0.125)]

    Returns:
        a tuple of Figure.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_figures() -> None:
    got = extract_figures("Opinion: 6,000 records at 22.6%, ECE 0.0727, change -0.0588.")
    assert all(isinstance(f, Figure) for f in got), "return a tuple of Figure"
    assert [f.text for f in got] == ["6,000", "22.6%", "0.0727", "-0.0588"], (
        f"figures came out {[f.text for f in got]} — '6,000' is ONE figure, and the sign "
        "and the percent sign belong to the figure they touch")
    assert all(abs(a - b) < 1e-12 for a, b in
              zip([f.value for f in got], [6000.0, 0.226, 0.0727, -0.0588])), (
        f"values {[f.value for f in got]} — commas removed, 22.6% means 0.226")
    assert abs(got[1].half_unit - 0.0005) < 1e-15 and got[0].half_unit == 0.5, (
        f"half units {[f.half_unit for f in got]} — '22.6%' claims 0.226 to within "
        "0.0005, and '6,000' to within 0.5")
    names = extract_figures("# Conformity pack -- D0007\n\n## 3. Findings\n"
                            "As of 2026-09-23, rated S1.")
    assert names == (), (
        f"found {[f.text for f in names]} in a document holding no results — dates, "
        "heading numbers and identifiers are names, not figures")
    ranged = extract_figures("| 0.00-0.10 | 0.0471 |")
    assert [f.value for f in ranged] == [0.0, 0.1, 0.0471], (
        f"got {[f.value for f in ranged]} — in '0.00-0.10' the dash follows a digit, so it "
        "is a range, not a minus sign")
    assert extract_figures("## 4. Limitations (5 found)")[0].value == 5.0, (
        "only the section number of a heading is a name; other numbers on it are figures")
    print("exercise 1 looks right")

In [ ]:
_try("exercise 1", _check_figures)

## 5. Exercise 2 — `computed_values()`

A figure traces to a *computed result* — so the gate needs every result a run produced,
with the path that locates it: `residual['money'].point`, `cost_figures[3].value`. Two
things in a pipeline's output are never results, and treating either as one is how a gate
quietly stops working. A **string** is not a result even when it is full of digits — the
commentary a person typed is a string, and if its digits counted, a typed number would
trace to itself. A **boolean** is not a result either: Python treats `True` as the
integer `1`, and a flag is not a count.

<details><summary>💡 Hint 1 — what to think about</summary>

This is a recursive walk, and each kind of container has its own idea of a name for its
children: a mapping has keys, a named tuple has field names, a plain list or tuple has
positions, a numpy array has indices. Order the type checks carefully — a named tuple IS
a tuple, and a bool IS an int, so the more specific test must come first. NaN and
infinity are never printed as a figure, so leave them out too.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Write an inner function of a node and its path so far. Return immediately for booleans
(Python's and numpy's), strings, bytes and `None`. For a Python or numpy integer or
float, convert to `float` and record it with its path when finite. For a numpy array,
skip a boolean array whole and otherwise walk every element with its flat index in
brackets. For an object with `_fields` (a named tuple), walk each field with a dot and
its name. For a mapping, walk each value with a dot and its key. For any other list or
tuple, walk each item with its position in brackets. A child of the root carries no
leading dot.

</details>

In [ ]:
def computed_values(obj: Any) -> tuple:
    """Every computed result inside `obj`, as `(path, value)` pairs, in walk order.

    Requirements, each graded:
      * a result is a Python or numpy integer or float, converted to `float`, and finite.
      * NOT results: `bool` and `numpy.bool_`, strings and bytes (even full of digits),
        `None`, NaN and the infinities, and anything of another type.
      * the walk is DEPTH FIRST and descends into mappings (values in insertion order),
        named tuples (fields in order), plain lists and tuples (items in order), and numpy
        arrays (every element in flat index order; a boolean array is skipped whole).
      * paths: a mapping key or a named-tuple field is joined with "." ("residual.money");
        a list, tuple or array position is in square brackets ("cost_figures[3].value",
        "arr[2]"). A child of the root has no leading ".".

    Example:
        >>> computed_values({"a": 1, "b": {"c": [0.5, True, "7"]}})
        (('a', 1.0), ('b.c[0]', 0.5))

    Returns:
        a tuple of (path, float) pairs.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_values() -> None:
    got = computed_values({"a": 1, "b": {"c": [0.5, True, "7"]}})
    assert isinstance(got, tuple) and got == (("a", 1.0), ("b.c[0]", 0.5)), (
        f"got {got} — the docstring example: True is a flag and '7' is a string, and "
        "neither is a computed result")
    iv = Interval(0.25, 0.20, 0.30, 0.04)
    paths = dict(computed_values({"residual": {"money": iv}, "n": np.int64(3),
                                  "gap": np.float64(-0.5), "flag": np.bool_(True),
                                  "mask": np.array([True, False]),
                                  "missing": float("nan")}))
    assert paths.get("residual.money.lo") == 0.20 and paths.get("residual.money.point") == 0.25, (
        f"paths found: {sorted(paths)} — a named tuple's fields are named with '.'")
    assert paths.get("n") == 3.0 and paths.get("gap") == -0.5, (
        "numpy integers and floats are results too")
    assert "flag" not in paths and not any(p.startswith("mask") for p in paths), (
        "numpy booleans and boolean arrays are flags, not results")
    assert "missing" not in paths, "NaN is never a figure anybody printed; leave it out"
    arr = np.array([[1.5, 2.5], [3.5, 4.5]])
    arr_paths = dict(computed_values({"m": arr}))
    assert arr_paths.get("m[1,0]") == 3.5, (
        f"paths found: {sorted(arr_paths)} — a 2-D array element is indexed '[i,j]'")
    print("exercise 2 looks right")

In [ ]:
_try("exercise 2", _check_values)

## 6. Exercise 3 — `trace_gate()`

Now the gate. For every figure in a document, look for a computed result within that
figure's own half unit — the precision the document itself claimed — and record the
first path that matches. A figure with no such result is **untraceable**, and one
untraceable figure fails the document.

Resist every temptation to make it pass more often. A gate that tolerates a bit more
than half a unit passes a figure copied one digit wrong. A gate that skips integers, or
percentages, or headings, or tables, passes a typed number of that kind. A gate whose
values include strings passes a typed number by finding it in the very sentence that
typed it.

<details><summary>💡 Hint 1 — what to think about</summary>

Everything you need already exists: `extract_figures` and `computed_values`. The only
work is the comparison, and the discipline is that the tolerance always comes from the
figure itself, never from a constant you choose.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Extract the figures and collect the values once. For each figure, take the first value
in `computed_values` order whose distance from the figure is no more than the figure's
`half_unit`. Record the figure with that value's path, or record it as untraced. The
document passes only when nothing is untraced.

</details>

In [ ]:
TRACE_SLACK = 1e-9         # the one allowance: binary floating point, not a wider tolerance.
                           # A result sitting exactly half a unit from its printed figure is
                           # legitimately rounded either way (0.03125 prints as "0.0312"
                           # under round-half-to-even, and 0.03125 is not exactly
                           # representable in binary), so the comparison allows one part in
                           # a billion of slack on the half unit itself — never a fixed or a
                           # relative tolerance on the figure's value.


class GateResult(NamedTuple):
    passed: bool
    n_figures: int
    traced: tuple            # ((Figure, path), ...) in document order
    untraced: tuple           # (Figure, ...) in document order


def trace_gate(report: str, evidence: Any) -> GateResult:
    """Trace every figure in `report` to a computed result in `evidence`, or fail it.

    Requirements, each graded:
      * a figure traces to a value when
        `abs(value - figure.value) <= figure.half_unit * (1 + TRACE_SLACK)`. Nothing
        wider: not a fixed tolerance, not a relative one on the value, not `half_unit`
        scaled up by anything other than `TRACE_SLACK`.
      * it traces to the FIRST such value in `computed_values(evidence)` order.
      * every figure is checked: integers, percentages, negatives, table cells, headings.
      * `passed` is True exactly when `untraced` is empty; `n_figures` counts every figure.

    Example:
        >>> r = trace_gate("F1 0.0727, n = 6,000", {"f1": 0.072691, "n": 6000})
        >>> r.passed, [path for _, path in r.traced]
        (True, ['f1', 'n'])

    Returns:
        a GateResult.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_gate() -> None:
    ok = trace_gate("F1 0.0727, n = 6,000", {"f1": 0.072691, "n": 6000})
    assert isinstance(ok, GateResult), "return a GateResult"
    assert ok.passed and [p for _, p in ok.traced] == ["f1", "n"], (
        f"the docstring example should pass, tracing to ['f1', 'n']; got passed={ok.passed}, "
        f"untraced {[f.text for f in ok.untraced]}")
    assert trace_gate("rate 22.6%", {"rate": 0.22598}).passed, (
        "'22.6%' means 0.226 and traces to 0.22598 — compare on the figure's own scale")
    slip = trace_gate("F1 0.0728", {"f1": 0.072691})
    assert not slip.passed and [f.text for f in slip.untraced] == ["0.0728"], (
        "0.072691 prints as 0.0727; a document saying 0.0728 is one digit wrong and must "
        "fail — is your tolerance wider than the figure's half unit?")
    typed = trace_gate("7 findings are open", {"open": 6, "note": "7 findings are open"})
    assert not typed.passed, (
        "a count of 7 that no computed result holds must fail, even though a string in the "
        "evidence says it — strings are not results, and integers are not exempt")
    assert not trace_gate("gap -0.0123", {"gap": 0.0123}).passed, (
        "-0.0123 does not trace to +0.0123: a sign is a direction")
    assert trace_gate("", {}).n_figures == 0, "an empty document has no figures to check"
    boundary = trace_gate("share 0.0312", {"share": 0.03125})
    assert boundary.passed, (
        "0.03125 legitimately prints as '0.0312' under round-half-to-even, and 0.03125 is "
        "not exactly representable in binary — this must trace with TRACE_SLACK, no wider "
        "tolerance needed")
    tight = trace_gate("value 1.00", {"v": 0.994})
    assert not tight.passed, (
        "'1.00' claims its value to within 0.005; 0.994 is 0.006 away — a tolerance any "
        "wider than half_unit (times TRACE_SLACK) would wrongly pass this")
    print("exercise 3 looks right")

In [ ]:
_try("exercise 3", _check_gate)

In [ ]:
def _show_gate_on_draft() -> None:
    claimed = draft_claimed_evidence_with_defects()
    result = trace_gate(DRAFT, claimed)
    print(f"the draft pack: {result.n_figures} figures, {len(result.traced)} traced, "
          f"{len(result.untraced)} untraceable")
    for fig in result.untraced:
        print(f"  UNTRACEABLE {fig.text!r} on line {fig.line}: nothing in the pack's own "
          "evidence bundle prints as this figure.")
    print("every OTHER figure traced — the residual table, the blended cost and the "
          "calibration day included, wrong as they are — because the preparer's own bundle "
          "is self-consistent. Reproducing a bundle the preparer wrote is not reproducing "
          "the raw artefacts behind it; sections 7 to 9 build the difference.")


_try("the gate on the draft pack", _show_gate_on_draft, needs=("exercise 3",))

## 7. Exercise 4 — `check_residual_provenance()`

The residual table above traced. It is still wrong: `DEFECT_RESIDUAL` is
`RESIDUAL_TRAIN`, labelled as the evaluation split. The fix is not a wider tolerance —
the numbers are exactly right, for the wrong data — it is recomputing the residual table
from the raw records, on the split the pack claims, and refusing to take the pack's word
for which split it used.

<details><summary>💡 Hint 1 — what to think about</summary>

You have `residual_by_class` and the raw `records`. Recompute the required split
yourself; do not trust `claimed`. If a claimed figure does not match your recomputed
eval figure, check whether it matches the OTHER split before giving up — that turns "this
number is wrong" into "this number was computed on the training split", which is the
useful thing to tell whoever prepared it.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Recompute `residual_by_class(records, required_split)`: this is your source of truth.
Also compute the other split, only to name the mistake. For each field type in
`claimed`, compare its `.point` to the required split's `.point` within a tight
tolerance (bootstrap intervals are stochastic; the point estimate is not, so an exact
comparison is correct here — do not derive a tolerance from the interval). A field whose
claimed point does not match belongs in `bad`; say in `detail` which split it actually
matches, or that it matches neither.

</details>

In [ ]:
class Audit(NamedTuple):
    passed: bool
    bad: tuple           # names of what is wrong: field types, figure labels, or a marker
    detail: tuple          # one explanatory message per entry of `bad`, in the same order


def check_residual_provenance(claimed: Mapping[str, Interval], records: Sequence[Mapping],
                              required_split: str = "eval", tol: float = 1e-9) -> Audit:
    """Recompute the residual table from `records` and confirm every claimed figure was
    computed on `required_split`, not merely labelled as it.

    Requirements, each graded:
      * recompute `residual_by_class(records, required_split)`: this is the only source of
        truth: never compare `claimed` against itself or trust its label.
      * a field type's claimed `.point` must equal the recomputed split's `.point` within
        `tol` (an exact float comparison up to `tol`, never derived from `.lo`/`.hi`).
      * a claimed field that fails is added to `bad`. Its `detail` message names which
        split it actually matches (`required_split`'s opposite, "train" or "eval") when
        one does, or says it matches neither.
      * `passed` is True exactly when `bad` is empty. Field types are checked in
        `FIELD_TYPES` order.

    Example:
        >>> check_residual_provenance(RESIDUAL_EVAL, RECORDS, "eval").passed
        True

    Returns:
        an Audit.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_residual_provenance() -> None:
    honest = check_residual_provenance(RESIDUAL_EVAL, RECORDS, "eval")
    assert isinstance(honest, Audit) and honest.passed and honest.bad == (), (
        f"RESIDUAL_EVAL, checked against 'eval', should pass; got bad={honest.bad}")
    caught = check_residual_provenance(RESIDUAL_TRAIN, RECORDS, "eval")
    assert not caught.passed and set(caught.bad) == set(FIELD_TYPES), (
        f"RESIDUAL_TRAIN, checked against the required 'eval' split, should fail on every "
        f"field type; got bad={caught.bad}")
    assert all("train split" in d for d in caught.detail), (
        f"the detail message should name the split the figure actually matches (train); "
        f"got {caught.detail}")
    fabricated = {ft: Interval(0.9, 0.85, 0.95, 0.01) for ft in FIELD_TYPES}
    neither = check_residual_provenance(fabricated, RECORDS, "eval")
    assert not neither.passed and all("matches neither" in d for d in neither.detail), (
        f"a residual of 0.9 on every field matches no split at all; got {neither.detail}")
    print("exercise 4 looks right")

In [ ]:
_try("exercise 4", _check_residual_provenance)

## 8. Exercise 5 — `check_cost_provenance()`

The blended cost figure in the draft pack is numerically correct — it is the same
`BLENDED_COST` your clean pack will print — but it is labelled `"measured"` with no
`source_id`. A number correctly computed from tiers is a **derived** figure, and this
lesson's `MEASUREMENTS` registry is where a **measured** one must be found. Presenting a
derived figure as measured, with nothing behind it, is the defect.

<details><summary>💡 Hint 1 — what to think about</summary>

`"derived"` figures are not this check's business — they are checked by `trace_gate`
against the evidence bundle instead, which is where a computed figure belongs. Only
`"measured"` figures need a citation here, and a citation is not enough on its own: the
registry entry it points to must actually say the same number.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Walk the figures. Skip anything `"derived"`. For a `"measured"` figure with no
`source_id`, or one not present in `measurements`, or one whose registered value differs
from the figure's, add it to `bad` with a message naming which of the three went wrong.

</details>

In [ ]:
def check_cost_provenance(figures: Sequence[CostFigure], measurements: Mapping[str, Measurement],
                          tol: float = 1e-9) -> Audit:
    """Confirm every `"measured"` cost figure cites a real, matching source.

    Requirements, each graded:
      * a figure with `kind != "measured"` (i.e. `"derived"`) is never checked here.
      * a `"measured"` figure with an empty `source_id` is bad: "no source_id".
      * a `"measured"` figure whose `source_id` is not a key of `measurements` is bad:
        the id is not in the registry.
      * a `"measured"` figure whose `source_id` IS in `measurements`, but whose `value`
        differs from `measurements[source_id].value` by more than `tol`, is bad: the
        citation does not back the number.
      * `bad` holds each failing figure's `label`, in the order the figures were given;
        `detail` holds one message per entry of `bad`, in the same order.

    Example:
        >>> ms = {"cost.cheap.rate": Measurement("cost.cheap.rate", "x", 0.02, "$/doc")}
        >>> fig = CostFigure("cheap", 0.02, "measured", "cost.cheap.rate")
        >>> check_cost_provenance((fig,), ms).passed
        True

    Returns:
        an Audit.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_cost_provenance() -> None:
    clean = check_cost_provenance(CLEAN_COST_FIGURES, MEASUREMENTS)
    assert isinstance(clean, Audit) and clean.passed, (
        f"every clean cost figure is either measured-with-a-real-source or derived; got "
        f"bad={clean.bad}")
    caught = check_cost_provenance(DEFECT_COST_FIGURES, MEASUREMENTS)
    assert not caught.passed and caught.bad == ("blended cost per document",), (
        f"the blended figure claims kind='measured' with no source_id; got bad={caught.bad}")
    assert "no source_id" in caught.detail[0], (
        f"the message should name the mistake as a missing source_id; got {caught.detail}")
    unknown = CostFigure("mystery", 1.0, "measured", "cost.does.not.exist")
    bad_id = check_cost_provenance((unknown,), MEASUREMENTS)
    assert not bad_id.passed and "not in the measurement registry" in bad_id.detail[0], (
        f"a source_id absent from the registry must fail with its own message; got "
        f"{bad_id.detail}")
    stale = CostFigure("cheap extractor rate", 0.05, "measured", "cost.cheap.rate")
    bad_value = check_cost_provenance((stale,), MEASUREMENTS)
    assert not bad_value.passed, (
        "a source_id that exists but whose registered value differs from the figure must "
        "still fail — a citation is not enough on its own")
    always_ok = CostFigure("whatever", 999.0, "derived", "")
    assert check_cost_provenance((always_ok,), MEASUREMENTS).passed, (
        "a derived figure is never this check's business, however implausible its value")
    print("exercise 5 looks right")

In [ ]:
_try("exercise 5", _check_cost_provenance)

## 9. Exercise 6 — `check_threshold_timing()`

`DEFECT_THRESHOLD.calibrated_through_day` is later than `ALARM.alarm_day` — the threshold
used to call that day an alarm was fixed *after* the alarm fired. Every
number involved is real and correctly computed; the defect is purely in their order. A
threshold calibrated with hindsight is not evidence of anything, because it was free to
be chosen so that the alarm it explains would fire.

<details><summary>💡 Hint 1 — what to think about</summary>

This is a single comparison between two days, not a recomputation — the numbers are not
in doubt, only their order. Decide whether the boundary itself (calibrated through the
exact day the alarm fired) should count as honest or as hindsight, and be consistent
with the reason a threshold has to predate what it explains.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

The calibration window must end strictly before the day being explained: a threshold
calibrated "through" the alarm day already had that day's data in hand, so equality
counts as hindsight, the same as any later day. When it passes, nothing is bad; when it
fails, there is one bad entry and one message naming both days.

</details>

In [ ]:
def check_threshold_timing(threshold: Threshold, alarm_day: int) -> Audit:
    """Confirm the threshold was calibrated strictly before the alarm it is used to justify.

    Requirements, each graded:
      * passes when `threshold.calibrated_through_day < alarm_day`; a calibration window
        that reaches the alarm day itself (`==`) has already seen that day's data and
        must fail, same as a window that reaches past it.
      * on failure, `bad` is `("threshold",)` and `detail` is a one-item tuple naming both
        days and explaining that the calibration window must end before the alarm it
        justifies.

    Example:
        >>> check_threshold_timing(Threshold(0.5, 6.0, 100), 150).passed
        True

    Returns:
        an Audit.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_threshold_timing() -> None:
    ok = check_threshold_timing(Threshold(0.5, 6.0, 100), 150)
    assert isinstance(ok, Audit) and ok.passed and ok.bad == (), (
        f"a threshold calibrated through day 100 for an alarm on day 150 is honest; got "
        f"bad={ok.bad}")
    boundary = check_threshold_timing(Threshold(0.5, 6.0, 150), 150)
    assert not boundary.passed, (
        "calibrated 'through' the alarm day already saw that day's data — the boundary "
        "must fail, not pass")
    late = check_threshold_timing(DEFECT_THRESHOLD, ALARM.alarm_day)
    assert not late.passed and late.bad == ("threshold",), (
        f"DEFECT_THRESHOLD was calibrated after the alarm it is used to justify; got "
        f"bad={late.bad}")
    honest = check_threshold_timing(CLEAN_THRESHOLD, ALARM.alarm_day)
    assert honest.passed, (
        f"CLEAN_THRESHOLD was calibrated {ALARM.alarm_day - CLEAN_THRESHOLD.calibrated_through_day} "
        "days before the alarm and should pass")
    print("exercise 6 looks right")

In [ ]:
_try("exercise 6", _check_threshold_timing)

## 10. Exercise 7 — `validate_pack()`

Put the four checks together. `trace_gate` catches a figure with nothing behind it at
all; the other three catch a figure that reproduces perfectly and is still wrong,
because it was measured on the wrong data, cited nothing real, or was tuned with
hindsight. A pack is documented only when every one of the four holds — this is an AND,
not a majority vote, and dropping any one of the four from the combination is exactly
the kind of "fix" that makes a checker quieter without making a pack honest.

<details><summary>💡 Hint 1 — what to think about</summary>

This exercise calls the other six; it introduces no new comparison of its own. The
discipline is entirely in not skipping one, and in building `findings` so that every
failure — from any of the four — is named specifically enough to act on.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Run `trace_gate`, `check_residual_provenance`, `check_cost_provenance` and
`check_threshold_timing` — every one of them, unconditionally. Build `findings` as a
flat tuple of strings: one line per untraced figure (naming its text and line), then
every `detail` message from the residual check, then every one from the cost check, then
every one from the timing check, each prefixed with which check raised it. `passed` is
the logical AND of all four `.passed` values.

</details>

In [ ]:
class PackAudit(NamedTuple):
    passed: bool
    reproduction: GateResult
    residual: Audit
    cost: Audit
    timing: Audit
    findings: tuple


def validate_pack(report: str, evidence: Any, claimed_residual: Mapping[str, Interval],
                  records: Sequence[Mapping], cost_figures: Sequence[CostFigure],
                  measurements: Mapping[str, Measurement], threshold: Threshold,
                  alarm_day: int) -> PackAudit:
    """Regenerate every number in `report` from the artefacts behind it, and fail the pack
    if any of the four checks does.

    Requirements, each graded:
      * runs all four checks unconditionally: `trace_gate(report, evidence)`,
        `check_residual_provenance(claimed_residual, records)`,
        `check_cost_provenance(cost_figures, measurements)`,
        `check_threshold_timing(threshold, alarm_day)`.
      * `passed` is True exactly when all four `.passed` are True — an AND over every
        check, never a majority and never a subset.
      * `findings` is a flat tuple of strings, in this order: one "REPRODUCTION: ..." line
        per untraced figure (its `.text` and `.line`), then one "RESIDUAL SPLIT: ..." line
        per residual `.detail` entry, then one "COST PROVENANCE: ..." line per cost
        `.detail` entry, then one "THRESHOLD TIMING: ..." line per timing `.detail` entry.

    Example:
        >>> validate_pack(_pack_body(RESIDUAL_EVAL, "eval", CLEAN_COST_FIGURES,
        ...                          CLEAN_THRESHOLD, ALARM.alarm_day, "No open items."),
        ...                _build_claimed_evidence(RESIDUAL_EVAL, CLEAN_COST_FIGURES,
        ...                                        CLEAN_THRESHOLD, ALARM.alarm_day),
        ...                RESIDUAL_EVAL, RECORDS, CLEAN_COST_FIGURES, MEASUREMENTS,
        ...                CLEAN_THRESHOLD, ALARM.alarm_day).passed
        True

    Returns:
        a PackAudit.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_validate_pack() -> None:
    clean_text = _pack_body(CLEAN_RESIDUAL, CLEAN_SPLIT_LABEL, CLEAN_COST_FIGURES,
                            CLEAN_THRESHOLD, ALARM.alarm_day, CLEAN_COMMENTARY)
    clean_evidence = _build_claimed_evidence(CLEAN_RESIDUAL, CLEAN_COST_FIGURES,
                                             CLEAN_THRESHOLD, ALARM.alarm_day)
    clean = validate_pack(clean_text, clean_evidence, CLEAN_RESIDUAL, RECORDS,
                          CLEAN_COST_FIGURES, MEASUREMENTS, CLEAN_THRESHOLD, ALARM.alarm_day)
    assert isinstance(clean, PackAudit) and clean.passed and clean.findings == (), (
        f"the clean pack should pass every check; got findings={clean.findings}")

    defect_text = draft_pack_with_defects()
    defect_evidence = draft_claimed_evidence_with_defects()
    defect = validate_pack(defect_text, defect_evidence, DEFECT_RESIDUAL, RECORDS,
                           DEFECT_COST_FIGURES, MEASUREMENTS, DEFECT_THRESHOLD, ALARM.alarm_day)
    assert not defect.passed, "the draft pack has four planted defects and must not pass"
    assert not defect.reproduction.passed, (
        "the commentary's typed coverage figure has nothing behind it: REPRODUCTION should "
        "fail")
    assert not defect.residual.passed, (
        "the residual table was computed on the training split: RESIDUAL SPLIT should fail")
    assert not defect.cost.passed, (
        "the blended cost figure claims to be measured with no source: COST PROVENANCE "
        "should fail")
    assert not defect.timing.passed, (
        "the threshold was calibrated after the alarm it explains: THRESHOLD TIMING should "
        "fail")
    assert len(defect.findings) >= 4, (
        f"all four defects should each contribute at least one finding; got "
        f"{len(defect.findings)}: {defect.findings}")

    # a pack with ONLY the cost defect must still fail overall — passed is an AND, not a
    # majority, and dropping the cost check from the combination is exactly the kind of
    # "fix" that makes validate_pack quieter without making a pack honest.
    only_cost_bad = validate_pack(clean_text, clean_evidence, CLEAN_RESIDUAL, RECORDS,
                                  DEFECT_COST_FIGURES, MEASUREMENTS, CLEAN_THRESHOLD,
                                  ALARM.alarm_day)
    assert not only_cost_bad.passed and only_cost_bad.reproduction.passed, (
        "one failing check among four passing ones must still fail the whole pack")
    print("exercise 7 looks right")

In [ ]:
_try("exercise 7", _check_validate_pack)

## 11. The artefact — `render_conformity_pack()`

The point of the whole exercise. It builds the pack's text exactly as `_pack_body` always
has, then runs `validate_pack` over it before handing anything back — and raises, naming
every finding, rather than return a pack it cannot vouch for. GIVEN: it calls your
`validate_pack`, and introduces no check of its own.

In [ ]:
def render_conformity_pack(residual: Mapping[str, Interval], split_label: str,
                           cost_figures: Sequence[CostFigure], threshold: Threshold,
                           alarm_day: int, commentary: str,
                           measurements: Mapping[str, Measurement] = MEASUREMENTS,
                           records: Sequence[Mapping] = RECORDS) -> str:
    """GIVEN: the honest artefact. Refuses to return a pack `validate_pack` does not pass."""
    text = _pack_body(residual, split_label, cost_figures, threshold, alarm_day, commentary)
    evidence = _build_claimed_evidence(residual, cost_figures, threshold, alarm_day)
    audit = validate_pack(text, evidence, residual, records, cost_figures, measurements,
                          threshold, alarm_day)
    if not audit.passed:
        raise ValueError("pack failed validation:\n- " + "\n- ".join(audit.findings))
    return text


def _show_two_packs() -> None:
    clean = render_conformity_pack(CLEAN_RESIDUAL, CLEAN_SPLIT_LABEL, CLEAN_COST_FIGURES,
                                   CLEAN_THRESHOLD, ALARM.alarm_day, CLEAN_COMMENTARY)
    print("CLEAN PACK renders:")
    print(clean[:220] + " ...")
    try:
        render_conformity_pack(DEFECT_RESIDUAL, DEFECT_SPLIT_LABEL, DEFECT_COST_FIGURES,
                               DEFECT_THRESHOLD, ALARM.alarm_day, DEFECT_COMMENTARY)
        raise AssertionError("the defective pack rendered — validate_pack let it through")
    except ValueError as exc:
        print("\nDEFECT PACK refused:")
        print(str(exc))


_try("the two packs", _show_two_packs, needs=("exercise 7",))

## 12. Common mistakes

- **Trusting the pack's own evidence bundle for everything.** `trace_gate` only proves
  that a figure matches *something the preparer says they computed* — that is necessary,
  not sufficient; the table below measures how many of this lesson's defects it misses.
  A real validator recomputes from raw artefacts wherever it can, which is exactly what
  `check_residual_provenance` does and `trace_gate` structurally cannot.
- **Deriving a tolerance from anything other than the figure.** A fixed tolerance of
  `0.0001`, or `half_unit * 2` "to be safe", both pass a figure copied one digit wrong —
  which is the entire class of error this gate exists to catch.
- **Treating a `"derived"` cost figure as needing a citation, or a `"measured"` one as
  not needing one.** Both mistakes are visible in `DEFECT_COST_FIGURES`: the same value,
  correctly computed, becomes a defect purely by being labelled `"measured"` with
  nothing behind it.
- **Comparing calibration and alarm days with `<=`.** A window that reaches the alarm
  day itself already had that day's data when the threshold was chosen; the comparison
  must be strict.
- **Combining the four checks with anything but AND.** A pack with three checks green
  and one red is not "mostly documented" — it is not documented, and `validate_pack`
  must say so every time, not only when it feels like it.

In [ ]:
def _measure_defect_matrix() -> None:
    """Plant each defect ALONE in an otherwise clean pack and record which checks fail."""
    alarm = ALARM.alarm_day
    single = {
        "a typed commentary figure": (CLEAN_RESIDUAL, CLEAN_COST_FIGURES, CLEAN_THRESHOLD,
                                      DEFECT_COMMENTARY),
        "a residual table on the wrong split": (DEFECT_RESIDUAL, CLEAN_COST_FIGURES,
                                                CLEAN_THRESHOLD, CLEAN_COMMENTARY),
        "a derived cost labelled measured": (CLEAN_RESIDUAL, DEFECT_COST_FIGURES,
                                             CLEAN_THRESHOLD, CLEAN_COMMENTARY),
        "a threshold calibrated after the alarm": (CLEAN_RESIDUAL, CLEAN_COST_FIGURES,
                                                   DEFECT_THRESHOLD, CLEAN_COMMENTARY),
    }
    checks = ("reproduction", "residual", "cost", "timing")
    print(f"{'planted defect, alone in a clean pack':<40}" + "".join(f"{c:>14}" for c in checks)
          + f"{'pack':>10}")
    by_gate = by_validator = 0
    for label, (res, cost, thr, comm) in single.items():
        # every pack claims the eval split; only the wrong-split one is not telling the truth
        text = _pack_body(res, CLEAN_SPLIT_LABEL, cost, thr, alarm, comm)
        evidence = _build_claimed_evidence(res, cost, thr, alarm)
        audit = validate_pack(text, evidence, res, RECORDS, cost, MEASUREMENTS, thr, alarm)
        marks = ["FAILS" if not getattr(audit, c).passed else "passes" for c in checks]
        by_gate += not audit.reproduction.passed
        by_validator += not audit.passed
        print(f"{label:<40}" + "".join(f"{m:>14}" for m in marks)
              + f"{'refused' if not audit.passed else 'ACCEPTED':>10}")
    print(f"trace_gate alone catches {by_gate} of {len(single)} planted defects; "
          f"validate_pack refuses {by_validator} of {len(single)}.")


_try("which check catches which defect", _measure_defect_matrix, needs=("exercise 7",))

## 13. Self-check

1. Every figure in a residual-error table passes `trace_gate`: each one matches a number
   in the preparer's own evidence bundle. What does that tell you about which split the
   table was computed on?
   - (a) nothing — the bundle records whatever was computed, on whichever split; only
     recomputing from the raw records, on the split the table names, can say
   - (b) that it was computed on the split its heading names, because a traced figure
     is by definition a documented one
   - (c) that the split label is honest, because a mislabelled table could not trace
2. A colleague "fixes" the draft pack by deleting the commentary sentence `trace_gate`
   flagged, re-runs `trace_gate` alone, and it now passes. What should you conclude?
   - (a) the pack is now documented: nothing left in it fails to reproduce
   - (b) only that nothing left in the text is untraceable against the preparer's own
     bundle; the wrong-split table, the unsourced cost figure and the hindsight
     threshold are all still there, and `validate_pack` still refuses the pack
   - (c) nothing: deleting a sentence can never change a gate's verdict
3. Why does `check_cost_provenance` skip every `"derived"` figure, rather than asking it
   for a `source_id` too?
   - (a) derived figures are unimportant and do not need checking
   - (b) `MEASUREMENTS` happens not to contain any derived figures
   - (c) a derived figure is a computed result, not a claimed measurement: `trace_gate`
     checks it against the evidence bundle, and demanding a citation it never claimed
     to have would invent a failure with no defect behind it
4. `check_threshold_timing` fails a threshold calibrated exactly through the alarm day
   (`calibrated_through_day == alarm_day`), not only one calibrated after it. Why?
   - (a) "through day N" means the calibration data includes day N, so that threshold
     was chosen with the alarm day's data already in hand — the hindsight the check
     exists to catch
   - (b) it is an arbitrary choice; either boundary would do
   - (c) integer days should always be compared with `<=` by convention

Answers come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
print(f"this pipeline: {N_DOCS} documents, {len(FIELD_TYPES)} field types, "
      f"{len(MEASUREMENTS)} registered cost measurements")

## What you built, and what the programme leaves you with

A validator that does not ask "is this pack green" but "does every number in it come
back the same way twice, from the artefacts that are supposed to back it" — which is the
only definition of "documented" a machine can check, and the reason a figure can be
numerically perfect and still fail. This is the last module of
the programme: the harness, the review policy, slice analysis, the drift monitor and the
cost model all feed one pack, and one pack is only as trustworthy as the checks that
would refuse it.

In [ ]:
def _progress_board() -> None:
    order = list(_EXERCISES)
    done = sum(1 for name in order if _STATUS.get(name) == "passed")
    icon = {"passed": "✅", "failed": "❌"}
    print("Progress")
    for name in order:
        mark = icon.get(_STATUS.get(name), "⏳")
        print(f"  {mark} {name} ({', '.join(_EXERCISES[name])})")
    print(f"{done} of {len(order)} complete")


# Re-run every check one last time before the board, so the board is current even if you
# edited an exercise and did not re-run its own check cell.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_figures), ("exercise 2", _check_values),
                              ("exercise 3", _check_gate),
                              ("exercise 4", _check_residual_provenance),
                              ("exercise 5", _check_cost_provenance),
                              ("exercise 6", _check_threshold_timing),
                              ("exercise 7", _check_validate_pack)):
            _try(_name, _check)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green
    # exit code paper over it. Inside a notebook kernel the board above has already said so,
    # in a line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.